In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Step 1: Load the Dataset
df = pd.read_csv("F:\\01. The Data Psychology\\4. New_Machine_Learning\\7. Random Forest\\22. Flight_price_clean.csv")

In [3]:
df.head()

,flight_id,departure_city,destination_city,airline,departure_hour,distance_km,booking_days,is_weekend,flight_duration_hours,demand_level,price_usd
0,1,Los Angeles,Denver,United,19,4509,30,0,5.1,High,311.29
1,2,New York,Miami,United,8,1285,3,0,2.0,Low,214.26
2,3,Miami,Houston,Southwest,19,4664,35,0,5.5,Low,203.16
3,4,Houston,Denver,Southwest,18,4444,71,0,4.7,High,297.55
4,5,Houston,Boston,United,19,4286,76,1,5.1,Medium,253.25


In [4]:
# Drop flight_id (not useful for prediction)
df = df.drop('flight_id', axis=1)

In [5]:
df.shape

(2000, 10)

In [6]:
df.dtypes

departure_city            object
destination_city          object
airline                   object
departure_hour             int64
distance_km                int64
booking_days               int64
is_weekend                 int64
flight_duration_hours    float64
demand_level              object
price_usd                float64
dtype: object

In [7]:
df.isnull().sum()

departure_city           0
destination_city         0
airline                  0
departure_hour           0
distance_km              0
booking_days             0
is_weekend               0
flight_duration_hours    0
demand_level             0
price_usd                0
dtype: int64

In [8]:
df.duplicated().sum()

0

In [9]:
df.describe()

,departure_hour,distance_km,booking_days,is_weekend,flight_duration_hours,price_usd
count,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,11.589500,3023.982500,45.919500,0.493000,3.563450,251.882090
std,6.996532,1143.790457,26.382741,0.500076,1.112258,106.802676
min,0.000000,1008.000000,1.000000,0.000000,2.000000,150.000000
25%,5.000000,2061.250000,23.000000,0.000000,2.600000,166.777500
50%,12.000000,3015.500000,46.000000,0.000000,3.500000,219.625000
75%,18.000000,4014.250000,69.000000,1.000000,4.500000,301.390000
max,23.000000,4998.000000,90.000000,1.000000,6.000000,600.000000


In [10]:
le_departure = LabelEncoder()
le_destination = LabelEncoder()
le_airline = LabelEncoder()
le_demand = LabelEncoder()

In [11]:
df['departure_city'] = le_departure.fit_transform(df['departure_city'])

In [12]:
df['departure_city']

0       5
1       7
2       6
3       4
4       4
       ..
1995    3
1996    7
1997    7
1998    5
1999    7
Name: departure_city, Length: 2000, dtype: int32

In [13]:
df['destination_city'] = le_destination.fit_transform(df['destination_city'])
df['airline'] = le_airline.fit_transform(df['airline'])
df['demand_level'] = le_demand.fit_transform(df['demand_level'])

In [14]:
df.head()

,departure_city,destination_city,airline,departure_hour,distance_km,booking_days,is_weekend,flight_duration_hours,demand_level,price_usd
0,5,3,4,19,4509,30,0,5.1,0,311.29
1,7,6,4,8,1285,3,0,2.0,1,214.26
2,6,4,3,19,4664,35,0,5.5,1,203.16
3,4,3,3,18,4444,71,0,4.7,0,297.55
4,4,1,4,19,4286,76,1,5.1,2,253.25


In [15]:
df['time_of_day'] = pd.cut(df['departure_hour'], 
                          bins=[0, 6, 12, 18, 24], 
                          labels=[0, 1, 2, 3],  # Night, Morning, Afternoon, Evening
                          include_lowest=True)

In [16]:
df.head()

,departure_city,destination_city,airline,departure_hour,distance_km,booking_days,is_weekend,flight_duration_hours,demand_level,price_usd,time_of_day
0,5,3,4,19,4509,30,0,5.1,0,311.29,3
1,7,6,4,8,1285,3,0,2.0,1,214.26,1
2,6,4,3,19,4664,35,0,5.5,1,203.16,3
3,4,3,3,18,4444,71,0,4.7,0,297.55,2
4,4,1,4,19,4286,76,1,5.1,2,253.25,3


In [17]:
df['distance_per_hour'] = df['distance_km'] / df['flight_duration_hours']

In [18]:
df.head()

,departure_city,destination_city,airline,departure_hour,distance_km,booking_days,is_weekend,flight_duration_hours,demand_level,price_usd,time_of_day,distance_per_hour
0,5,3,4,19,4509,30,0,5.1,0,311.29,3,884.117647
1,7,6,4,8,1285,3,0,2.0,1,214.26,1,642.500000
2,6,4,3,19,4664,35,0,5.5,1,203.16,3,848.000000
3,4,3,3,18,4444,71,0,4.7,0,297.55,2,945.531915
4,4,1,4,19,4286,76,1,5.1,2,253.25,3,840.392157


In [19]:
df = df.drop('departure_hour', axis=1)

In [20]:
X = df.drop('price_usd', axis=1)
y = df['price_usd']

In [21]:
X.head()

,departure_city,destination_city,airline,distance_km,booking_days,is_weekend,flight_duration_hours,demand_level,time_of_day,distance_per_hour
0,5,3,4,4509,30,0,5.1,0,3,884.117647
1,7,6,4,1285,3,0,2.0,1,1,642.500000
2,6,4,3,4664,35,0,5.5,1,3,848.000000
3,4,3,3,4444,71,0,4.7,0,2,945.531915
4,4,1,4,4286,76,1,5.1,2,3,840.392157


In [22]:
y

0       311.29
1       214.26
2       203.16
3       297.55
4       253.25
         ...  
1995    258.22
1996    340.37
1997    150.00
1998    161.26
1999    168.54
Name: price_usd, Length: 2000, dtype: float64

In [23]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [24]:
rf = RandomForestRegressor(random_state=42)

In [25]:
rf.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [30]:
rf.score(X_test,y_test)

0.9794674528847119

In [31]:
y_pred = rf.predict(X_test)

mse = mean_squared_error(y_test, y_pred)

rmse = np.sqrt(mse)

r2 = r2_score(y_test, y_pred)

print(f"Root Mean Squared Error: {rmse:.2f}")
print(f"R² Score: {r2:.2f}")

Root Mean Squared Error: 15.06
R² Score: 0.98


In [32]:
param_dist = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']  # Removed 'auto' as it's deprecated
}

In [33]:
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring='neg_mean_squared_error',  # or 'neg_root_mean_squared_error'
    n_jobs=-1,
    random_state=42,
    verbose=2  # Optional: to see progress
)

In [34]:
random_search.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


,estimator,RandomForestR...ndom_state=42)
,param_distributions,"{'max_depth': [10, 20, ...], 'max_features': ['sqrt', 'log2'], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], ...}"
,n_iter,20
,scoring,'neg_mean_squared_error'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [35]:
best_rf = random_search.best_estimator_

print("Best Hyperparameters:", random_search.best_params_)

Best Hyperparameters: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': None}


In [36]:
y_train_pred = best_rf.predict(X_train)

# Predict on test data
y_test_pred = best_rf.predict(X_test)

In [37]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Training metrics
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_mae = mean_absolute_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

# Testing metrics
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_mae = mean_absolute_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("Training RMSE:", train_rmse)
print("Training MAE:", train_mae)
print("Training R²:", train_r2)

print("Testing RMSE:", test_rmse)
print("Testing MAE:", test_mae)
print("Testing R²:", test_r2)


Training RMSE: 8.851606995121097
Training MAE: 6.1343974583333445
Training R²: 0.9931814480098519
Testing RMSE: 24.569610161088804
Testing MAE: 17.510421166666685
Testing R²: 0.945328584365465
